# Train Kikuyu VITS From Scratch (WaxalNLP `kik_tts`)

This notebook executes the full pipeline in Colab:
1. Environment setup
2. Dataset preprocessing + speaker-disjoint manifests
3. Tokenizer/vocab build
4. From-scratch training with resume-safe checkpointing
5. Evaluation + checkpoint selection
6. Local integration artifact packaging


In [ ]:
!pip install -U pip
!pip install datasets[audio] soundfile librosa pyyaml huggingface_hub
!pip install coqpit trainer
!pip install TTS==0.22.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!git clone https://github.com/kihahu/kikuyu-tts.git
%cd /content/kikuyu-tts


In [ ]:
!python scripts/prepare_waxal_kik_tts.py \
  --dataset-name google/WaxalNLP \
  --dataset-config kik_tts \
  --split train \
  --output-dir data/waxal_kik_tts \
  --target-sample-rate 16000 \
  --min-duration-sec 0.6 \
  --max-duration-sec 25.0 \
  --min-rms 0.0035 \
  --seed 42 \
  --dev-ratio 0.10 \
  --test-ratio 0.05


In [ ]:
!python scripts/build_kikuyu_vocab.py \
  --train-manifest data/waxal_kik_tts/manifests/train.jsonl \
  --dev-manifest data/waxal_kik_tts/manifests/dev.jsonl \
  --out-dir artifacts/tokenizer_kikuyu_char


In [ ]:
%cd /content
!git clone https://github.com/coqui-ai/TTS.git
%cd /content/kikuyu-tts


In [ ]:
# Start fresh
!python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/TTS


In [ ]:
# Resume mode
!python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/TTS \
  --resume


In [ ]:
# Optional: push checkpoints to HF Hub
!huggingface-cli login
!python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/TTS \
  --resume \
  --push-hf


Create a metrics CSV at `artifacts/checkpoint_metrics.csv` with columns:
- `checkpoint`
- `synthesis_success_rate`
- `clipping_rate`
- `mos_lite`
- `wer_proxy`


In [ ]:
!python scripts/evaluate_and_select.py \
  --metrics-csv artifacts/checkpoint_metrics.csv \
  --out-json artifacts/best_checkpoint_selection.json \
  --out-csv artifacts/tts_eval_summary.csv


In [ ]:
!python scripts/prepare_local_integration.py \
  --best-checkpoint-dir artifacts/colab_runs/kikuyu_vits_scratch/checkpoint_best \
  --tokenizer-dir artifacts/tokenizer_kikuyu_char \
  --eval-summary-csv artifacts/tts_eval_summary.csv \
  --out-dir artifacts/local_integration/kikuyu_vits_best \
  --model-id kikuyu-vits-scratch-waxal
